# Colab VSCode SSH Worker
Run this notebook to start an SSH daemon on your Google Colab instance, tunneled through Cloudflare. 
This allows you to securely connect your local VSCode directly to the Colab GPU.

In [ ]:
import os
import subprocess
import random
import string

# 1. Install OpenSSH Server
!apt-get install -y openssh-server > /dev/null 2>&1
!mkdir -p /var/run/sshd

# 2. Generate secure random password
password = ''.join(random.choices(string.ascii_letters + string.digits, k=16))
!echo 'root:'{password} | chpasswd

# 3. Configure SSHD to allow root login
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config

# 4. Start the SSH daemon
!/usr/sbin/sshd
print("✅ SSH Daemon started.")

In [ ]:
# 5. Install Cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Cloudflared installed.")

In [ ]:
import threading
import time

# 6. Start Cloudflare Tunnel in the background
def start_cloudflared():
    os.system("cloudflared tunnel --url ssh://localhost:22 > cloudflared.log 2>&1")

threading.Thread(target=start_cloudflared, daemon=True).start()
time.sleep(3)

# 7. Parse the unique TryCloudflare URL
try:
    with open("cloudflared.log", "r") as f:
        logs = f.read()
        for line in logs.split('\n'):
            if "trycloudflare.com" in line:
                url = line[line.find("https://") + 8:line.find(".trycloudflare.com") + 18]
                print("\n" + "="*60)
                print("🚀 YOUR SSH TUNNEL IS LIVE!")
                print("="*60)
                print(f"Host URL: {url}")
                print(f"Password: {password}")
                print("\nUpdate your VSCode ~/.ssh/config or orchestrator with this URL.")
                break
except Exception as e:
    print(f"Failed to parse Cloudflare URL: {e}. Check cloudflared.log manually.")